In [5]:
"""
Ensemble Method 1 — SOFT VOTING  (extended)
Models : XGBoost x2, LightGBM x2, CatBoost x2, ExtraTrees,
         RandomForest, GradientBoosting, SVM (RBF+Poly),
         LogisticRegression (L1+L2), KNN x2
Variants: equal-weight | AUROC-weighted | rank-average | optimised-weight
"""

import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.stats import rankdata

from sklearn.ensemble import (
    ExtraTreesClassifier, RandomForestClassifier, GradientBoostingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    roc_auc_score, matthews_corrcoef, accuracy_score, classification_report,
)
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier
import warnings
warnings.filterwarnings("ignore")

# ── LOAD DATA ──────────────────────────────────────────────────────────────────
BASE = r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf"

X_train = pd.read_csv(f"{BASE}\\X_train_rF_DEG_train.csv", low_memory=False).values.astype(float)
X_test  = pd.read_csv(f"{BASE}\\X_test_rF_DEG_train.csv",  low_memory=False).values.astype(float)
y_train = pd.read_csv(f"{BASE}\\y_train_rF_DEG_train.csv",  low_memory=False)["cancer_type"].values.astype(int)
y_test  = pd.read_csv(f"{BASE}\\y_test_rF_DEG_train.csv",   low_memory=False)["cancer_type"].values.astype(int)



In [6]:
# ── BASE MODELS ────────────────────────────────────────────────────────────────
BASE_MODELS = {

    "XGBoost_v1": xgb.XGBClassifier(
        n_estimators=500, max_depth=6, learning_rate=0.03,
        subsample=0.8, colsample_bytree=0.7, colsample_bylevel=0.7,
        min_child_weight=3, gamma=0.1, reg_alpha=0.1, reg_lambda=1.5,
        use_label_encoder=False, eval_metric="logloss", random_state=42, n_jobs=-1,
    ),
    "XGBoost_v2": xgb.XGBClassifier(
        n_estimators=500, max_depth=9, learning_rate=0.02,
        subsample=0.7, colsample_bytree=0.6, min_child_weight=1,
        gamma=0.2, reg_alpha=0.5, reg_lambda=2.0,
        use_label_encoder=False, eval_metric="logloss", random_state=7, n_jobs=-1,
    ),

    "LightGBM_v1": lgb.LGBMClassifier(
        n_estimators=500, max_depth=7, learning_rate=0.03,
        num_leaves=63, subsample=0.8, colsample_bytree=0.7,
        min_child_samples=20, reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbose=-1,
    ),
    "LightGBM_dart": lgb.LGBMClassifier(
        boosting_type="dart", n_estimators=300, max_depth=6,
        learning_rate=0.05, num_leaves=31, subsample=0.8,
        drop_rate=0.1, skip_drop=0.5,
        random_state=42, n_jobs=-1, verbose=-1,
    ),

    "CatBoost_v1": CatBoostClassifier(
        iterations=500, depth=6, learning_rate=0.03,
        l2_leaf_reg=3.0, border_count=128, bagging_temperature=0.5,
        random_strength=1.0, random_seed=42, verbose=0,
    ),
    "CatBoost_v2": CatBoostClassifier(
        iterations=500, depth=9, learning_rate=0.02,
        l2_leaf_reg=5.0, border_count=64, bagging_temperature=1.0,
        random_strength=2.0, random_seed=7, verbose=0,
    ),

    "ExtraTrees": ExtraTreesClassifier(
        n_estimators=500, max_depth=None, min_samples_split=4,
        min_samples_leaf=2, max_features="sqrt",
        bootstrap=False, random_state=42, n_jobs=-1,
    ),
    "RandomForest": RandomForestClassifier(
        n_estimators=500, max_depth=None, min_samples_split=4,
        min_samples_leaf=2, max_features="sqrt",
        bootstrap=True, oob_score=True, random_state=42, n_jobs=-1,
    ),
    "GradientBoosting": GradientBoostingClassifier(
        n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.8, min_samples_split=4,
        max_features="sqrt", random_state=42,
    ),

    "SVM_RBF": Pipeline([
        ("sc", StandardScaler()),
        ("clf", SVC(kernel="rbf", C=10.0, gamma="scale",
                    probability=True, random_state=42)),
    ]),
    "SVM_Poly": Pipeline([
        ("sc", StandardScaler()),
        ("clf", SVC(kernel="poly", degree=3, C=5.0, coef0=1.0,
                    probability=True, random_state=42)),
    ]),

    "LR_L2": Pipeline([
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(penalty="l2", C=1.0, solver="lbfgs",
                                    max_iter=2000, random_state=42, n_jobs=-1)),
    ]),
    "LR_L1": Pipeline([
        ("sc", StandardScaler()),
        ("clf", LogisticRegression(penalty="l1", C=0.5, solver="saga",
                                    max_iter=2000, random_state=42, n_jobs=-1)),
    ]),

    "KNN_5": Pipeline([
        ("sc", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=5, weights="distance",
                                      metric="minkowski", n_jobs=-1)),
    ]),
    "KNN_15": Pipeline([
        ("sc", StandardScaler()),
        ("clf", KNeighborsClassifier(n_neighbors=15, weights="distance",
                                      metric="euclidean", n_jobs=-1)),
    ]),
}

# ── HELPERS ───────────────────────────────────────────────────────────────────
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate(y_true, y_prob, name):
    y_pred = (y_prob >= 0.5).astype(int)
    print(f"\n{'='*60}\n  {name}\n{'='*60}")
    print(f"  AUROC : {roc_auc_score(y_true, y_prob):.4f}")
    print(f"  MCC   : {matthews_corrcoef(y_true, y_pred):.4f}")
    print(f"  Acc   : {accuracy_score(y_true, y_pred):.4f}")
    print(classification_report(y_true, y_pred,
                                 target_names=["Non-cancer (0)", "Cancer (1)"]))

In [7]:


# ── TRAIN ALL MODELS ──────────────────────────────────────────────────────────
print("Training base models ...\n")
test_probas = {}
cv_aurocs   = {}
oof_matrix  = np.zeros((len(y_train), len(BASE_MODELS)))

for col, (name, model) in enumerate(BASE_MODELS.items()):
    model.fit(X_train, y_train)
    p = model.predict_proba(X_test)[:, 1]
    test_probas[name] = p

    oof = cross_val_predict(model, X_train, y_train,
                             cv=CV, method="predict_proba")[:, 1]
    oof_matrix[:, col] = oof
    cv_aurocs[name]    = roc_auc_score(y_train, oof)
    print(f"  {name:25s} | CV-AUROC={cv_aurocs[name]:.4f} | "
          f"test μ={p.mean():.4f}")

proba_matrix = np.column_stack(list(test_probas.values()))

# ── VOTING VARIANT 1: Equal weight ───────────────────────────────────────────
equal_proba = proba_matrix.mean(axis=1)
evaluate(y_test, equal_proba, "Soft Voting — Equal Weights")

# ── VOTING VARIANT 2: AUROC-weighted ─────────────────────────────────────────
auroc_vals  = np.array([cv_aurocs[n] for n in test_probas])
auroc_w     = auroc_vals / auroc_vals.sum()
auroc_proba = proba_matrix @ auroc_w
evaluate(y_test, auroc_proba, "Soft Voting — AUROC-Weighted")

# ── VOTING VARIANT 3: Rank-average ───────────────────────────────────────────
rank_matrix = np.apply_along_axis(
    lambda col: rankdata(col) / len(col), 0, proba_matrix
)
rank_proba = rank_matrix.mean(axis=1)
evaluate(y_test, rank_proba, "Soft Voting — Rank-Average")

# ── VOTING VARIANT 4: Optimised weight (Nelder-Mead on OOF AUROC) ────────────
print("\nOptimising ensemble weights (Nelder-Mead) ...")

def neg_auroc(w):
    w = np.abs(w); w /= w.sum() + 1e-9
    return -roc_auc_score(y_train, oof_matrix @ w)

n_m    = len(BASE_MODELS)
result = minimize(neg_auroc, np.ones(n_m) / n_m, method="Nelder-Mead",
                  options={"maxiter": 10000, "xatol": 1e-7})
opt_w  = np.abs(result.x); opt_w /= opt_w.sum()

print("\n  Optimised weights:")
for name, w in zip(BASE_MODELS.keys(), opt_w):
    print(f"    {name:25s} : {w:.4f}")

opt_proba = proba_matrix @ opt_w
evaluate(y_test, opt_proba, "Soft Voting — Optimised Weights (Nelder-Mead)")

# ── SUMMARY TABLE ─────────────────────────────────────────────────────────────
variants = {
    "Equal-weight":     equal_proba,
    "AUROC-weighted":   auroc_proba,
    "Rank-average":     rank_proba,
    "Optimised-weight": opt_proba,
}
print("\n" + "="*60)
print("  SOFT VOTING — SUMMARY")
print(f"  {'Variant':<22} {'AUROC':>7} {'MCC':>7} {'Acc':>7}")
print("="*60)
for vn, vp in variants.items():
    vpr = (vp >= 0.5).astype(int)
    print(f"  {vn:<22} "
          f"{roc_auc_score(y_test, vp):>7.4f} "
          f"{matthews_corrcoef(y_test, vpr):>7.4f} "
          f"{accuracy_score(y_test, vpr):>7.4f}")
print()

Training base models ...

  XGBoost_v1                | CV-AUROC=0.9267 | test μ=0.8412
  XGBoost_v2                | CV-AUROC=0.9234 | test μ=0.8465
  LightGBM_v1               | CV-AUROC=0.9189 | test μ=0.8594
  LightGBM_dart             | CV-AUROC=0.9130 | test μ=0.8265
  CatBoost_v1               | CV-AUROC=0.9232 | test μ=0.8485
  CatBoost_v2               | CV-AUROC=0.9154 | test μ=0.8485
  ExtraTrees                | CV-AUROC=0.8995 | test μ=0.8058
  RandomForest              | CV-AUROC=0.8930 | test μ=0.8066
  GradientBoosting          | CV-AUROC=0.9150 | test μ=0.8768
  SVM_RBF                   | CV-AUROC=0.9320 | test μ=0.7816
  SVM_Poly                  | CV-AUROC=0.9217 | test μ=0.7878
  LR_L2                     | CV-AUROC=0.9015 | test μ=0.7873
  LR_L1                     | CV-AUROC=0.9202 | test μ=0.8014
  KNN_5                     | CV-AUROC=0.8489 | test μ=0.7511
  KNN_15                    | CV-AUROC=0.8839 | test μ=0.7637

  Soft Voting — Equal Weights
  AUROC : 0.9

# Ver 2


In [8]:
"""
Ensemble Method — EXPANDED MULTI-PARAMETER OPTIMIZED BAYESIAN MODEL AVERAGING
Models: XGBoost, LightGBM, Logistic Regression, Random Forest, Extra Trees, SVM (RBF)
Features: Full hyperparameter tuning for every base model + ensemble alpha via Optuna.
"""

import numpy as np
import pandas as pd
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score, matthews_corrcoef
import xgboost as xgb
import lightgbm as lgb
import warnings

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── 1. LOAD DATA ──────────────────────────────────────────────────────────────
BASE = r"C:\Users\chaulh\Downloads\Trang_tre_trau\paper\minh\paper 2026\Gene Expression Omnibus (GEO) Dataset GSE68086\python_desque2\Binary\data\ranked rf"

def load_numeric_features(filepath):
    df = pd.read_csv(filepath, low_memory=False)
    return df.select_dtypes(include=[np.number]).values.astype(float)

X_train = load_numeric_features(f"{BASE}\\X_train_rF_DEG_train.csv")
X_test  = load_numeric_features(f"{BASE}\\X_test_rF_DEG_train.csv")
y_train = pd.read_csv(f"{BASE}\\y_train_rF_DEG_train.csv", low_memory=False)["cancer_type"].values.astype(int)
y_test  = pd.read_csv(f"{BASE}\\y_test_rF_DEG_train.csv",  low_memory=False)["cancer_type"].values.astype(int)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [12]:


# ── 2. MODEL FACTORY ──────────────────────────────────────────────────────────
def build_models(trial):
    """
    Returns a dict of {name: sklearn-compatible model} using parameters
    sampled by the Optuna trial object.
    """
    # ── Ensemble sharpening parameter ───────────────────────────────────────
    alpha = trial.suggest_float("alpha", 1.0, 20.0)

    # ── XGBoost ─────────────────────────────────────────────────────────────
    xgb_params = dict(
        n_estimators      = trial.suggest_int  ("xgb_n_estimators",  100, 400, step=50),
        learning_rate     = trial.suggest_float("xgb_lr",            0.01, 0.2,  log=True),
        max_depth         = trial.suggest_int  ("xgb_max_depth",     3, 9),
        subsample         = trial.suggest_float("xgb_subsample",     0.5, 1.0),
        colsample_bytree  = trial.suggest_float("xgb_colsample",     0.5, 1.0),
        # min_child_weight  = trial.suggest_int  ("xgb_min_child_w",   1, 10),
        gamma             = trial.suggest_float("xgb_gamma",         0.0, 5.0),
        # reg_alpha         = trial.suggest_float("xgb_reg_alpha",     1e-4, 10.0, log=True),
        # reg_lambda        = trial.suggest_float("xgb_reg_lambda",    1e-4, 10.0, log=True),
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )
    xgboost_model = xgb.XGBClassifier(**xgb_params)

    # ── LightGBM ─────────────────────────────────────────────────────────────
    lgb_params = dict(
        n_estimators      = trial.suggest_int  ("lgb_n_estimators",  100, 400, step=50),
        learning_rate     = trial.suggest_float("lgb_lr",            0.01, 0.2,  log=True),
        max_depth         = trial.suggest_int  ("lgb_max_depth",     3, 9),
        num_leaves        = trial.suggest_int  ("lgb_num_leaves",    20, 150),
        # subsample         = trial.suggest_float("lgb_subsample",     0.5, 1.0),
        colsample_bytree  = trial.suggest_float("lgb_colsample",     0.5, 1.0),
        min_child_samples = trial.suggest_int  ("lgb_min_child_s",   5, 50),
        # reg_alpha         = trial.suggest_float("lgb_reg_alpha",     1e-4, 10.0, log=True),
        # reg_lambda        = trial.suggest_float("lgb_reg_lambda",    1e-4, 10.0, log=True),
        random_state=42, n_jobs=-1, verbose=-1,
    )
    lightgbm_model = lgb.LGBMClassifier(**lgb_params)

    # ── Logistic Regression ───────────────────────────────────────────────────
    lr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C          = trial.suggest_float("lr_C",       1e-3, 100.0, log=True),
            solver     = trial.suggest_categorical("lr_solver", ["lbfgs", "saga"]),
            max_iter   = 2000,
            random_state=42, n_jobs=-1,
        )),
    ])

    # ── Random Forest ─────────────────────────────────────────────────────────
    rf_model = RandomForestClassifier(
        n_estimators      = trial.suggest_int  ("rf_n_estimators",   100, 400, step=50),
        max_depth         = trial.suggest_int  ("rf_max_depth",      3, 20),
        min_samples_split = trial.suggest_int  ("rf_min_split",      2, 20),
        min_samples_leaf  = trial.suggest_int  ("rf_min_leaf",       1, 10),
        # max_features      = trial.suggest_categorical("rf_max_feat", ["sqrt", "log2", None]),
        # bootstrap         = trial.suggest_categorical("rf_bootstrap", [True, False]),
        random_state=42, n_jobs=-1,
    )

    # ── Extra Trees ───────────────────────────────────────────────────────────
    et_model = ExtraTreesClassifier(
        n_estimators      = trial.suggest_int  ("et_n_estimators",   100, 400, step=50),
        max_depth         = trial.suggest_int  ("et_max_depth",      3, 20),
        min_samples_split = trial.suggest_int  ("et_min_split",      2, 20),
        min_samples_leaf  = trial.suggest_int  ("et_min_leaf",       1, 10),
        # max_features      = trial.suggest_categorical("et_max_feat", ["sqrt", "log2", None]),
        random_state=42, n_jobs=-1,
    )

    # ── SVM (RBF kernel) ─────────────────────────────────────────────────────
    svm_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C          = trial.suggest_float("svm_C",     1e-2, 100.0, log=True),
            gamma      = trial.suggest_float("svm_gamma", 1e-4, 1.0,   log=True),
            kernel     = "rbf",
            probability=True,
            random_state=42,
        )),
    ])

    models = {
        "XGBoost"            : xgboost_model,
        "LightGBM"           : lightgbm_model,
        "LogisticRegression" : lr_model,
        "RandomForest"       : rf_model,
        "ExtraTrees"         : et_model,
        "SVM_RBF"            : svm_model,
    }
    return alpha, models


In [13]:

# ── 3. OBJECTIVE FUNCTION ─────────────────────────────────────────────────────
def objective(trial):
    alpha, models = build_models(trial)

    raw_weights = {}
    oof_probas  = {}

    for name, model in models.items():
        oof_proba = cross_val_predict(
            model, X_train, y_train, cv=CV, method="predict_proba"
        )[:, 1]
        oof_probas[name]  = oof_proba
        cv_auroc          = roc_auc_score(y_train, oof_proba)
        raw_weights[name] = np.exp(alpha * cv_auroc)

    # Normalized BMA blend
    total_w      = sum(raw_weights.values())
    oof_bma      = sum((raw_weights[n] / total_w) * oof_probas[n] for n in raw_weights)

    # Best threshold search → maximize OOF MCC
    best_mcc = -1.0
    for t in np.linspace(0.2, 0.8, 61):
        mcc = matthews_corrcoef(y_train, (oof_bma >= t).astype(int))
        if mcc > best_mcc:
            best_mcc = mcc

    return best_mcc



In [15]:
# ── 4. RUN OPTIMIZATION ───────────────────────────────────────────────────────
print("Starting Expanded Multi-Parameter Optimization (6 models + ensemble alpha)...")
print("Models: XGBoost, LightGBM, Logistic Regression, Random Forest, Extra Trees, SVM-RBF\n")

study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5),
)
study.optimize(objective, n_trials=50, show_progress_bar=True)
# ↑ Increase n_trials (100–200) for a thorough search in production.

print("\n🏆 Optimization Complete!")
print(f"   Best OOF MCC : {study.best_value:.4f}")
print("\n   Best Parameters Found:")
for k, v in sorted(study.best_params.items()):
    print(f"     {k:30s} = {v}")

# ── 5. RETRAIN FINAL ENSEMBLE WITH BEST PARAMS ────────────────────────────────
best = study.best_params

def build_final_models(best):
    """Rebuild models from best params dict. Uses .get(key, default) so
    any commented-out parameters in build_models() are safely skipped."""
    
    xgboost_model = xgb.XGBClassifier(
        n_estimators     = best.get("xgb_n_estimators", 200),
        learning_rate    = best.get("xgb_lr",           0.05),
        max_depth        = best.get("xgb_max_depth",    6),
        subsample        = best.get("xgb_subsample",    0.8),
        colsample_bytree = best.get("xgb_colsample",    0.8),
        min_child_weight = best.get("xgb_min_child_w",  1),      # default=1
        gamma            = best.get("xgb_gamma",        0.0),    # default=0
        reg_alpha        = best.get("xgb_reg_alpha",    0.0),    # default=0
        reg_lambda       = best.get("xgb_reg_lambda",   1.0),    # default=1
        eval_metric="logloss", random_state=42, n_jobs=-1,
    )
    lightgbm_model = lgb.LGBMClassifier(
        n_estimators      = best.get("lgb_n_estimators", 200),
        learning_rate     = best.get("lgb_lr",            0.05),
        max_depth         = best.get("lgb_max_depth",     6),
        num_leaves        = best.get("lgb_num_leaves",    31),    # default=31
        subsample         = best.get("lgb_subsample",     0.8),
        colsample_bytree  = best.get("lgb_colsample",     0.8),
        min_child_samples = best.get("lgb_min_child_s",   20),   # default=20
        reg_alpha         = best.get("lgb_reg_alpha",     0.0),
        reg_lambda        = best.get("lgb_reg_lambda",    1.0),
        random_state=42, n_jobs=-1, verbose=-1,
    )
    lr_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(
            C      = best.get("lr_C",      1.0),
            solver = best.get("lr_solver", "lbfgs"),
            max_iter=2000, random_state=42, n_jobs=-1,
        )),
    ])
    rf_model = RandomForestClassifier(
        n_estimators      = best.get("rf_n_estimators", 200),
        max_depth         = best.get("rf_max_depth",    None),   # default=None
        min_samples_split = best.get("rf_min_split",   2),
        min_samples_leaf  = best.get("rf_min_leaf",    1),
        max_features      = best.get("rf_max_feat",    "sqrt"),
        bootstrap         = best.get("rf_bootstrap",   True),
        random_state=42, n_jobs=-1,
    )
    et_model = ExtraTreesClassifier(
        n_estimators      = best.get("et_n_estimators", 200),
        max_depth         = best.get("et_max_depth",    None),
        min_samples_split = best.get("et_min_split",   2),
        min_samples_leaf  = best.get("et_min_leaf",    1),
        max_features      = best.get("et_max_feat",    "sqrt"),
        random_state=42, n_jobs=-1,
    )
    svm_model = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", SVC(
            C     = best.get("svm_C",     1.0),
            gamma = best.get("svm_gamma", "scale"),
            kernel="rbf", probability=True, random_state=42,
        )),
    ])
    return {
        "XGBoost"            : xgboost_model,
        "LightGBM"           : lightgbm_model,
        "LogisticRegression" : lr_model,
        "RandomForest"       : rf_model,
        "ExtraTrees"         : et_model,
        "SVM_RBF"            : svm_model,
    }

FINAL_MODELS = build_final_models(best)
alpha_final  = best["alpha"]

final_raw_weights  = {}
final_oof_probas   = {}
final_test_probas  = {}

print("\nRetraining final ensemble on full training data...")
for name, model in FINAL_MODELS.items():
    # OOF pass → for final weight calculation
    oof_p = cross_val_predict(model, X_train, y_train, cv=CV, method="predict_proba")[:, 1]
    final_oof_probas[name]  = oof_p
    final_raw_weights[name] = np.exp(alpha_final * roc_auc_score(y_train, oof_p))

    # Fit on full training set → predict test
    model.fit(X_train, y_train)
    final_test_probas[name] = model.predict_proba(X_test)[:, 1]
    print(f"  ✔ {name}")

# Final normalized BMA blends
total_final_w    = sum(final_raw_weights.values())
final_norm_w     = {k: v / total_final_w for k, v in final_raw_weights.items()}
final_oof_blend  = sum(final_norm_w[n] * final_oof_probas[n]  for n in final_norm_w)
final_test_blend = sum(final_norm_w[n] * final_test_probas[n] for n in final_norm_w)

# Threshold calibration on OOF blend
final_threshold  = 0.5
best_final_mcc   = -1.0
for t in np.linspace(0.1, 0.9, 81):
    mcc = matthews_corrcoef(y_train, (final_oof_blend >= t).astype(int))
    if mcc > best_final_mcc:
        best_final_mcc  = mcc
        final_threshold = t

# ── 6. RESULTS ────────────────────────────────────────────────────────────────
y_pred = (final_test_blend >= final_threshold).astype(int)

print("\n" + "=" * 60)
print("  FINAL ENSEMBLE RESULTS (Expanded BMA)")
print("=" * 60)
print(f"  Model Weights (normalized):")
for name, w in sorted(final_norm_w.items(), key=lambda x: -x[1]):
    print(f"    {name:22s}: {w:.4f}  (raw exp-weight: {final_raw_weights[name]:.2f})")
print(f"\n  Decision Threshold   : {final_threshold:.2f}")
print(f"  Test AUROC           : {roc_auc_score(y_test, final_test_blend):.4f}")
print(f"  Test MCC             : {matthews_corrcoef(y_test, y_pred):.4f}")
print("=" * 60)

Starting Expanded Multi-Parameter Optimization (6 models + ensemble alpha)...
Models: XGBoost, LightGBM, Logistic Regression, Random Forest, Extra Trees, SVM-RBF



  0%|          | 0/50 [00:00<?, ?it/s]


🏆 Optimization Complete!
   Best OOF MCC : 0.6577

   Best Parameters Found:
     alpha                          = 14.700189879316378
     et_max_depth                   = 11
     et_min_leaf                    = 3
     et_min_split                   = 14
     et_n_estimators                = 400
     lgb_colsample                  = 0.6243393476847732
     lgb_lr                         = 0.07699403787090768
     lgb_max_depth                  = 6
     lgb_min_child_s                = 25
     lgb_n_estimators               = 350
     lgb_num_leaves                 = 37
     lr_C                           = 0.25776654639112134
     lr_solver                      = saga
     rf_max_depth                   = 13
     rf_min_leaf                    = 10
     rf_min_split                   = 4
     rf_n_estimators                = 300
     svm_C                          = 0.0579947086184011
     svm_gamma                      = 0.032912852714050984
     xgb_colsample                  = 0.9

In [16]:
from sklearn.metrics import classification_report
# Reslut for testing set
from sklearn.metrics import roc_curve, auc


# print("Training time:", training_time, "seconds")
print(classification_report( y_test,y_pred,digits=6))


              precision    recall  f1-score   support

           0   0.702703  0.666667  0.684211        78
           1   0.921212  0.932515  0.926829       326

    accuracy                       0.881188       404
   macro avg   0.811957  0.799591  0.805520       404
weighted avg   0.879025  0.881188  0.879987       404

